# Project 2: Mission to Mars

Double tap this cell to get to edit mode and fill in the information below.

> Name of team:

> Team members (full names):

> Date:

> Resources used:

> Contributions of each team member:


## Tips

  * Python tutorials:
     * [A short introduction](https://realpython.com/python-first-steps/)
     * [A more complete introduction](https://www.w3schools.com/python/default.asp)
  * Use __esc r__ to disable a cell
  * Use __esc y__ to reactivate it
  * Use __esc m__ to go to markdown mode. **Markdown** is the typesetting language used in jupyter notebooks.
  * In a markdown cell, double tap the mouse or glide pad (on your laptop) to go to edit mode. 
  * Shift + return to execute a cell (including markdown cells).
  * If the equations don't typeset, try double tapping the cell again, and re-execute it.

## Introduction

A mission to Mars from the Earth requires a complicated series of calculations and a complicated series of steps [click here, for example](https://mars.nasa.gov/mer/mission/timeline/launch/launch-diagram/). But in this project, we'll ignore all of the complications and assume that at launch the probe is at the same position as the Earth and the probe's initial velocity is simply the sum of the Earth's velocity and the velocity of the probe relative to Earth ($\Delta v$ in NASA jargon), which we presume is provided by a short burn of the rocket attached to the probe.

## Project Goal

Your goal is to launch a space probe from Earth so that it arrives at Mars, by which we mean within **100,000 km** of Mars, such that its $\Delta v$ relative to Mars is also bounded. Of course, the closer you get the probe to Mars the better!

### Assumptions

  1. The probe starts at the same position as the Earth. 
  1. The probe's initial velocity, $\vec{u}$, **relative to Earth** points in the **same direction** as Earth's velocity, $\vec{v}_E$.
  1. The speed $u = |\vec{u}|$ satisfies the constraint $0 \lt u \leq 6\,\text{km / s}$ both at the Earth and at Mars. As noted, NASA refers to the magnitude of $\vec{u}$ as $\Delta v$. 
  1. The mass of the space probe plus rocket is $m_p = 2 \times 10^4 \, \text{kg}$ after the rocket burn.
  1. The probe continues along an orbit under the influence of the net gravitational force of all the inner planets, Jupiter and the Sun.
  2. The Sun is initially at the origin. 

To solve this problem you need to:
  1. choose an initial velocity, $\vec{u}$, relative to Earth;
  1. choose when to launch the space probe;
  1. compute the orbit of the probe, and
  1. compute the **minimum distance**, $d$, between the probe and Mars.
  1. If $d < 10^5 \, \text{km}$, the probe can be considered to have arrived in the vicinity of Mars.
  1. compute $\Delta \vec{v}$ and check that its magnitude, $\Delta v$, is less than 6 km/s.
  1. If not, repeat the above.  

## Suggestions of quantities to report, compute, and/or plot
  1. Your probe's initial $\Delta v$ in km/s.
  1. Time in Earth days for the probe to go from Earth to Mars.
  1. Distance traveled by the probe, in km.
  2. Plot of the Sun's speed in m/s versus time.
  1. Your probe's $\Delta v$ (in km/s) at Mars. Approximate velocities at time $t$ as follows
  
 \begin{align}
 \vec{v}(t) & \approx \frac{\vec{r}(t + \Delta t) - \vec{r}(t)}{\Delta t}.
 \end{align}

## Coordinate system
The coordinate system is displayed for that $+z$ points upwards and the $x-y$ plane is horizontal. 

In [1]:
import os, sys
import vpython as vp
import numpy as np

# We'll use this to read in the planetary data, which are in CSV files.
import pandas as pd

# We'll use this to update the date in the animation.
from datetime import date, timedelta

import matplotlib as mp
import matplotlib.pyplot as plt
FONTSIZE = 12
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : FONTSIZE}
mp.rc('font', **font)

# use latex if available on system, otherwise set usetex=False
import shutil
mp.rc('text', usetex=shutil.which('latex') is not None)
# --------------------------------------------------------------------
from comphyslab.graphics import Scene, CoordinateSystem, Sim, \
Controls, Zoom, J, K

from comphyslab.newton  import \
Gravity, propagate_order3,\
G, DAY, YEAR, AU2METERS, \
Msun, Mmercury, Mvenus, Mearth, Mmars, Mjupiter

from comphyslab.vectors import magnitude, unit, dot
from comphyslab.utils   import Bag

/Users/harry/miniconda3/envs/comphys/lib/python3.13/site-packages/vpython/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

## PART 1: Simulation

The function `read_position_data` reads the positions of the planets and the Sun in a heliocentric coordinate system and returns an array of position vectors in **Cartesian** $(x, y, z)$ coordinates. 

In [2]:
def read_position_data(filename, n=5):

    # Read data for Mercury through to Jupiter
    data = pd.read_csv(filename)[:n]

    # Convert to numpy arrays
    phi = data['longitude_deg'].to_numpy()    
    radius = data['distance_au'].to_numpy()

    # Convert from degrees to radians    
    phi = np.deg2rad(phi)

    # Compute positions in Cartesian coordinates
    x = radius * np.cos(phi)
    y = radius * np.sin(phi)

    # Assume planets lie in the same plane (approximately true!)
    z = np.zeros(len(x))

    # Insert coordinates of the Sun before those of the planets.
    # For simplicity assume that on 2026-03-30, the Sun is at 
    # the center of the solar system. 
    x = np.insert(x, 0, 0)
    y = np.insert(y, 0, 0)
    z = np.insert(z, 0, 0)
    r = np.array([x, y, z]).T
    
    return r

In [3]:
!ls data

planets_2026-03-30_midnight_EST.csv planets_2026-03-31_midnight_EST.csv


In [4]:
def define_initial_state():
    
    bg = Bag()
    
    bg.text  = ['Sun',  'Mercury', 'Venus',  'Earth', 'Mars', 'Jupiter']
    bg.mass  = np.array([Msun, Mmercury, Mvenus, Mearth, Mmars, Mjupiter])
    bg.color = ['yellow', 'magenta', 'white', 'cyan',  'red', 'orange']
    
    # Create vpython color objects from strings
    bg.vpcolor = [eval(f'vp.color.{color}') for color in bg.color]
        
    # Specify force law
    bg.law   = Gravity
    
    # Positions of the Sun, inner planets and Jupiter on 2026-03-30.
    bg.r  = read_position_data('data/planets_2026-03-30_midnight_EST.csv')
    bg.r0 = np.zeros_like(bg.r) # needed in update
    
    # Positions of the Sun, inner planets and Jupiter one Earth day later.
    bg.r_one_day_later = read_position_data(
        'data/planets_2026-03-31_midnight_EST.csv')
    
    # Time step
    bg.dt = DAY / 2 
    
    # Convert distances to SI units
    bg.r  *= AU2METERS
    bg.r_one_day_later *= AU2METERS

    # Approximate initial veloctities in m/s
    bg.v   = (bg.r_one_day_later - bg.r)/DAY
    bg.v0  = np.zeros_like(bg.v) # needed in update

    return bg

## Build the scene

In [5]:
def build_scene(bg):
    # ----------------------------------------------------
    # Required attributes
    # ----------------------------------------------------
    bg.rate   = 30         # No faster than "rate" frames/second
    bg.frame  = 0          # Frame counter
    bg.active = True       # Controled by Stop button 
    bg.update = False      # Controled by Start/Pause button

    # ----------------------------------------------------
    # Create empty scene
    # ----------------------------------------------------
    # All scene widgets must be placed in bag.gfx so that 
    # they can be properly deleted when the animation is 
    # stopped.
    gfx = bg.gfx           # give bag.gfx a shorter name

    bg.size = 2*AU2METERS  # Scale of scene in meters
    
    gfx.scene = Scene(
        'Mission to Mars\n', bg.size, up=K, height=300)

    # Create a Cartesian coordinate system with z-axis upwards
    gfx.xyz = CoordinateSystem(
        bg.size, up=K, draw_plane=False)

    # ----------------------------------------------------
    # Add scene elements
    # ----------------------------------------------------

    # Model each planet as a small sphere
    gfx.particles = []
    particle_radius = 0.04 * bg.size
    
    for i in range(len(bg.r)):
        if i == 0:
            radius = 2 * particle_radius # For the Sun
        else:
            radius = particle_radius
            
        gfx.particles.append(vp.sphere(
            pos=vp.vector(*bg.r[i]), 
            color=bg.vpcolor[i],
            radius=radius, 
            make_trail=i>0)) # Add trail

    # Add control buttons (Stop, Start/Pause)
    controls = Controls(bg)

    gfx.b_stop  = vp.button(
        text="Stop",
        background=vp.color.red,
        pos=gfx.scene.title_anchor,
        bind=controls.stop)

    gfx.b_start_pause = vp.button(
        text="Start",
        background=vp.color.green,
        pos=gfx.scene.title_anchor,
        bind=controls.start_pause)

    # Add zoom slider
    gfx.zoom = Zoom(gfx.scene)

    # ------------------------------------------------------
    # Add a launch button and a label to display the date 
    # ------------------------------------------------------
    # Create a button to launch the probe
    gfx.b_launch = vp.button(
        text="Launch",
        background=vp.color.white,
        pos=gfx.scene.title_anchor,
        bind=launch)
    # The launch button needs access to the bag, so
    # cache the bag in the button.
    gfx.b_launch.bg = bg

    # Create a label to display the date
    # We want the label to remain in a fixed position, so
    # we need to work in screen space, not world space.
    # Note: In vpython, the bottom-left corner is the
    # (x,y) origin of the pixel coordinates (0,0,0).
    bg.start_date  = date(2026, 3, 30)
    
    gfx.date_label = vp.label(
        text=str(bg.start_date),  # Convert date object to a string
        screen=True,              # Make label in screen space, not world space
        pixel_pos=True,           # Use pixel coordinates, not world coordinates
        pos=vp.vector(10, 20, 0), # Pixels from bottom-left: (x, y)
        align='left',             # pos pertains to the left of the label
        height=16,                # Font size in pixels
        color=vp.color.white,
        opacity=0,                # 0 = transparent background box
        border=0,
        box=False,
        line=False
    )

# ------------------------------------------------------------------
# Launch callback. This presumably is one of the places where
# you'll do some coding!
# ------------------------------------------------------------------
def launch(b):
    bg = b.bg # Need bag, which we cached in the button!
    
    elapsed_days = bg.frame * bg.dt / DAY
    
    current_date = str(bg.start_date + timedelta(days=elapsed_days))

    # Change button
    b.text = f'Launched: {current_date}'
    b.background = vp.color.green
    
    # Change the initial velocity of the probe!

In [6]:
def propagate(bg):
    # ----------------------------------------------
    # Update state of every particle.
    # ----------------------------------------------
    bg.r0[:] = bg.r # copy r into r0 before update
    bg.v0[:] = bg.v # copy v into v0 before update

    # Solve Newton's 2nd law for every particle
    bg.r[:], bg.v[:], bg.U = propagate_order3(
        -G, bg.mass, bg.mass, bg.r0, bg.v0, bg.law, bg.dt)

In [7]:
def update(bg):

    # Compute next positions of all particles
    propagate(bg)
    
    for i, particle in enumerate(bg.gfx.particles):
        x, y, z = bg.r[i]
        particle.pos.x = x
        particle.pos.y = y
        particle.pos.z = z

    # Update date
    # Note:
    #  dt  is in seconds 
    #  DAY is in seconds
    
    elapsed_days = bg.frame * bg.dt / DAY
    
    current_date = bg.start_date + timedelta(days=elapsed_days)

    # Update text in date label
    bg.gfx.date_label.text = str(current_date) # Must onvert date object to a string

In [8]:
bag = define_initial_state()

build_scene(bag)

sim = Sim(bag, update)

sim.run()

<IPython.core.display.Javascript object>

Animation ended!


## PART 2: Study, Results, and Conclusions